In [6]:
import os
import cv2   
import time
import shutil
import random
import numpy as np 
from tqdm import tqdm 
from glob import glob 
from PIL import Image 
import tensorflow as tf    
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
base_path = r"D:\final year project\all datasets\Augmentation_approach\Second Set"
preprocessed_path1 = r"D:\final year project\all datasets\Augmentation_approach\resized_set"
preprocessed_path2 = r"D:\final year project\all datasets\Augmentation_approach\denoised_set"
preprocessed_path3 = r"D:\final year project\all datasets\Augmentation_approach\enhanced_set" 
preprocessed_path4 = r"D:\final year project\all datasets\Augmentation_approach\augmented_balanced_set"
os.makedirs(preprocessed_path1, exist_ok=True)
os.makedirs(preprocessed_path2, exist_ok=True)
os.makedirs(preprocessed_path3, exist_ok=True)
os.makedirs(preprocessed_path4, exist_ok=True)

In [3]:
# RESIZE AUGMENTED IMAGES

TARGET_SIZE = (256, 256)

def resize_dataset(input_dir, output_dir):
    for class_name in ["400x Normal Oral Cavity Histopathological Images", "400x OSCC Histopathological Images"]:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)
        input_path = os.path.join(input_dir, class_name)
        output_path = os.path.join(output_dir, class_name)
        
        img_files = [f for f in os.listdir(input_path) if f.lower().endswith('.jpg')]
        
        for img_file in tqdm(img_files, desc=f"Resizing {class_name}"):
            try:
                img = cv2.imread(os.path.join(input_path, img_file))
                if img is not None:
                    resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                    cv2.imwrite(os.path.join(output_path, img_file), resized)
            except Exception:
                continue

print("Resizing images...")
resize_dataset(base_path, preprocessed_path1)
print(f"\nResizing complete. Output saved to: {preprocessed_path1}")

Resizing images...


Resizing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:32<00:00,  6.18it/s]
Resizing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [01:15<00:00,  6.51it/s]


Resizing complete. Output saved to: D:\final year project\all datasets\Augmentation_approach\resized_set


In [4]:
# Apply median filter to color image
def median_filter_color(img, kernel_size=3):
    # Ensure kernel size is odd and >= 3
    kernel_size = max(3, kernel_size | 1)
    return cv2.medianBlur(img, kernel_size)

# Denoise function using median filter
def denoise_dataset(input_dir, output_dir, kernel_size=3):
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all class folders
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    
    for class_name in tqdm(class_folders, desc="Processing classes"):
        class_input_dir = os.path.join(input_dir, class_name)
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Get all images in class folder
        image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg'))]
        
        for img_file in tqdm(image_files, desc=f"Processing {class_name}", leave=False):
            img_path = os.path.join(class_input_dir, img_file)
            
            # Load image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}")
                continue
            
            # Apply median filter
            filtered_img = median_filter_color(img, kernel_size=kernel_size)
            
            # Save result
            output_path = os.path.join(class_output_dir, img_file)
            cv2.imwrite(output_path, filtered_img)

# Example usage
if __name__ == "__main__":
    denoise_dataset(preprocessed_path1, preprocessed_path2, kernel_size=3)

Processing classes:   0%|          | 0/2 [00:00<?, ?it/s]

Processing classes: 100%|██████████| 2/2 [00:15<00:00,  7.95s/it]


In [5]:
#CLAHE

# Create output directory structure
class_dirs = [d for d in os.listdir(preprocessed_path2) if os.path.isdir(os.path.join(preprocessed_path2, d))]
for class_dir in class_dirs:
    os.makedirs(os.path.join(preprocessed_path3, class_dir), exist_ok=True)

# Apply CLAHE using OpenCV
def apply_clahe_to_image(image_np):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.7, tileGridSize=(8,8))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# Process all images
for class_dir in class_dirs:
    input_class_path = os.path.join(preprocessed_path2, class_dir)
    output_class_path = os.path.join(preprocessed_path3, class_dir)

    image_paths = glob(os.path.join(input_class_path, "*.jpg"))

    for image_path in tqdm(image_paths, desc=f"Processing {class_dir}"):
 
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.uint8)

        image_np = image.numpy()
        
        # Apply CLAHE
        clahe_image = apply_clahe_to_image(image_np)

        # Save output
        image_name = os.path.basename(image_path)
        output_path = os.path.join(output_class_path, image_name)
        cv2.imwrite(output_path, cv2.cvtColor(clahe_image, cv2.COLOR_RGB2BGR)) 

Processing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:05<00:00, 35.50it/s]
Processing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:11<00:00, 42.31it/s]


In [8]:
# Set the seed for reproducibility
random.seed(42)

# Paths
SOURCE_DIR = preprocessed_path3
DEST_DIR = preprocessed_path4

# Splits
SPLITS = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
}

# Create split folders
for split in SPLITS:
    for class_name in os.listdir(SOURCE_DIR):
        class_path = os.path.join(SOURCE_DIR, class_name)
        if os.path.isdir(class_path):
            split_dir = os.path.join(DEST_DIR, split, class_name)
            os.makedirs(split_dir, exist_ok=True)

# Process each class
for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg"))]
    random.shuffle(images)

    total = len(images)
    train_end = int(SPLITS["train"] * total)
    val_end = train_end + int(SPLITS["val"] * total)

    split_data = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    # Copy files with progress bar
    for split, split_images in split_data.items():
        print(f"\nCopying {split} data for class '{class_name}':")
        for img_name in tqdm(split_images, desc=f"{class_name} - {split}", unit="img"):
            src_path = os.path.join(class_path, img_name)
            dest_path = os.path.join(DEST_DIR, split, class_name, img_name)
            shutil.copy2(src_path, dest_path)

print("\n✅ Dataset split completed successfully.")


Copying train data for class '400x Normal Oral Cavity Histopathological Images':


















400x Normal Oral Cavity Histopathological Images - train: 100%|██████████| 140/140 [00:01<00:00, 80.62img/s] 



Copying val data for class '400x Normal Oral Cavity Histopathological Images':





400x Normal Oral Cavity Histopathological Images - val: 100%|██████████| 30/30 [00:00<00:00, 132.57img/s]



Copying test data for class '400x Normal Oral Cavity Histopathological Images':


...


Copying val data for class '400x OSCC Histopathological Images':







400x OSCC Histopathological Images - val: 100%|██████████| 74/74 [00:00<00:00, 111.61img/s]



Copying test data for class '400x OSCC Histopathological Images':






400x OSCC Histopathological Images - test: 100%|██████████| 75/75 [00:00<00:00, 193.87img/s]


✅ Dataset split completed successfully.


In [9]:
# ----------------------- CONFIGURATION -----------------------

# Dataset paths
INPUT_DIR = r"D:\final year project\all datasets\Augmentation_approach\augmented_balanced_set\train"
OUTPUT_DIR = r"D:\final year project\all datasets\Augmentation_approach\augmented_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output class folders
classes = [
    "400x Normal Oral Cavity Histopathological Images",
    "400x OSCC Histopathological Images"
]

for cls in classes:
    os.makedirs(os.path.join(OUTPUT_DIR, cls), exist_ok=True)

# Target count per class
TARGET_COUNT = 5000
BATCH_SIZE = 1

# ----------------------- AUGMENTATION SETUP -----------------------

# rotation, width and height shift, horizontal and vertical flip, contrast, brightness

datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True
)

def random_contrast_brightness(image):
    alpha = np.random.uniform(0.7, 1.3)
    beta = np.random.uniform(-7, 7)
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)


# ----------------------- AUGMENTATION FUNCTION -----------------------

def augment_class(class_name, target_count):
    input_path = os.path.join(INPUT_DIR, class_name)
    output_path = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(output_path, exist_ok=True)

    original_images = [
        os.path.join(input_path, f)
        for f in os.listdir(input_path)
        if f.lower().endswith('.jpg')
    ]
    
    existing_augmented = [
        f for f in os.listdir(output_path)
        if f.lower().endswith('.jpg')
    ]

    current_count = len(existing_augmented)
    remaining = target_count - current_count

    if current_count == 0:
        print(f"[INFO] Copying original images for {class_name}")
        for orig_img_path in original_images:
            try:
                img = Image.open(orig_img_path).convert('RGB')
                base_name = os.path.basename(orig_img_path)
                save_path = os.path.join(output_path, f"orig_{base_name}")
                img.save(save_path)
            except Exception as e:
                print(f"[ERROR] Failed to copy image {orig_img_path}: {e}")
        current_count = len(os.listdir(output_path))
        remaining = target_count - current_count

    if remaining <= 0:
        print(f"[INFO] {class_name}: Already has {current_count} images.")
        return

    print(f"\n[INFO] Augmenting '{class_name}' from {current_count} to {target_count} images")
    img_index = 0
    pbar = tqdm(total=remaining, desc=f"Augmenting {class_name}")

    while remaining > 0:
        img_path = original_images[img_index % len(original_images)]

        try:
            with Image.open(img_path).convert('RGB') as img:
                img_array = np.expand_dims(np.array(img), axis=0).astype(np.float32)
                num_augmentations = min(remaining, np.random.randint(4, 7))  # 4 to 6 augmentations

                for i, batch in enumerate(datagen.flow(img_array, batch_size=1)):
                    if i >= num_augmentations:
                        break

                    augmented_img = batch[0]
                    augmented_img = np.uint8(augmented_img)
                    final_img = random_contrast_brightness(augmented_img)

                    filename = f"aug_{os.path.splitext(os.path.basename(img_path))[0]}_{i}_{int(time.time())}.jpg"
                    save_path = os.path.join(output_path, filename)
                    Image.fromarray(final_img).save(save_path)

                    remaining -= 1
                    pbar.update(1)

                    if remaining <= 0:
                        break

        except Exception as e:
            print(f"[ERROR] Failed to process {img_path}: {e}")

        img_index += 1

    pbar.close()
    print(f"[DONE] {class_name}: Total images = {len(os.listdir(output_path))}")

# ----------------------- RUN -----------------------

for cls in classes:
    augment_class(cls, TARGET_COUNT)

# ----------------------- STATS -----------------------

print(f"\n[SUCCESS] Augmentation complete. Output saved to: {OUTPUT_DIR}")
for cls in classes:
    count = len(os.listdir(os.path.join(OUTPUT_DIR, cls)))
    print(f"- {cls}: {count} images")


[INFO] Copying original images for 400x Normal Oral Cavity Histopathological Images

[INFO] Augmenting '400x Normal Oral Cavity Histopathological Images' from 140 to 5000 images


[DONE] 400x Normal Oral Cavity Histopathological Images: Total images = 5000
[INFO] Copying original images for 400x OSCC Histopathological Images

[INFO] Augmenting '400x OSCC Histopathological Images' from 346 to 5000 images


[DONE] 400x OSCC Histopathological Images: Total images = 5000

[SUCCESS] Augmentation complete. Output saved to: D:\final year project\all datasets\Augmentation_approach\augmented_dataset
- 400x Normal Oral Cavity Histopathological Images: 5000 images
- 400x OSCC Histopathological Images: 5000 images
